# Archive Stratified Dataset Split

This notebook creates and archives the stratified breast-level split for VinDr-Mammo dataset.

**Output:**
- `vindr_mammo_stratified_split.zip` containing:
  - Original CSV file
  - Train indices (image-level)
  - Validation indices (image-level)
  - Split metadata (statistics, random seed, etc.)
  - Train/validation sample information

In [ ]:
import os
import json
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

## Configuration

In [ ]:
# Paths - ADJUST THESE TO YOUR SETUP
VINDR_CSV = "path/to/vindr_mammo.csv"  # Update this path
VINDR_IMAGES_ROOT = "path/to/vindr_images"  # Update this path
OUTPUT_ZIP = "vindr_mammo_stratified_split.zip"

# Split parameters (from CLAUDE.md)
TRAIN_RATIO = 0.8
RANDOM_STATE = 42
BENIGN_BIRADS = [1, 2, 3]
MALIGNANT_BIRADS = [5, 6]

## Step 1: Load Dataset and Create Stratified Split

In [ ]:
# Import dataset utilities
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from breast_cancer_detection.src.datasets import (
    VinDRMammoBinaryDataset,
    create_breast_level_splits,
    map_birads_to_binary
)
from breast_cancer_detection.src.preprocessing import MammographyPreprocessor

In [ ]:
# Create preprocessor (not needed for split creation, but required for dataset initialization)
preprocessor = MammographyPreprocessor(
    target_size=(720, 480),
    aspect_ratio=1.5
)

# Load full dataset
print("Loading VinDr-Mammo dataset...")
full_dataset = VinDRMammoBinaryDataset(
    images_root=VINDR_IMAGES_ROOT,
    csv_file=VINDR_CSV,
    preprocessor=preprocessor,
    transform=None,
    benign_birads=BENIGN_BIRADS,
    malignant_birads=MALIGNANT_BIRADS
)

print(f"\nTotal samples: {len(full_dataset)}")

In [ ]:
# Create stratified breast-level split
print("\nCreating stratified breast-level split...")
train_subset, val_subset = create_breast_level_splits(
    dataset=full_dataset,
    train_ratio=TRAIN_RATIO,
    random_state=RANDOM_STATE,
    stratify=True
)

# Extract indices
train_indices = train_subset.indices
val_indices = val_subset.indices

print(f"\nTrain indices: {len(train_indices)}")
print(f"Validation indices: {len(val_indices)}")

## Step 2: Prepare Split Information

In [ ]:
# Create metadata
split_metadata = {
    "created_at": datetime.now().isoformat(),
    "train_ratio": TRAIN_RATIO,
    "random_state": RANDOM_STATE,
    "benign_birads": BENIGN_BIRADS,
    "malignant_birads": MALIGNANT_BIRADS,
    "total_samples": len(full_dataset),
    "train_samples": len(train_indices),
    "val_samples": len(val_indices),
    "split_type": "breast_level_stratified",
    "description": "Patient-wise stratified split at breast level (CC+MLO views kept together)"
}

# Compute label distribution
train_labels = [full_dataset.samples[i][4] for i in train_indices]
val_labels = [full_dataset.samples[i][4] for i in val_indices]

split_metadata["train_label_counts"] = {
    "benign": int(np.sum(np.array(train_labels) == 0)),
    "malignant": int(np.sum(np.array(train_labels) == 1))
}

split_metadata["val_label_counts"] = {
    "benign": int(np.sum(np.array(val_labels) == 0)),
    "malignant": int(np.sum(np.array(val_labels) == 1))
}

print("\nSplit Metadata:")
print(json.dumps(split_metadata, indent=2))

In [ ]:
# Create train and validation sample DataFrames
train_samples_df = pd.DataFrame(
    [full_dataset.samples[i] for i in train_indices],
    columns=["study_id", "image_id", "laterality", "view_position", "label"]
)

val_samples_df = pd.DataFrame(
    [full_dataset.samples[i] for i in val_indices],
    columns=["study_id", "image_id", "laterality", "view_position", "label"]
)

print("\nTrain samples preview:")
print(train_samples_df.head())

print("\nValidation samples preview:")
print(val_samples_df.head())

## Step 3: Create Archive

In [ ]:
# Create temporary directory for files to archive
temp_dir = Path("temp_archive")
temp_dir.mkdir(exist_ok=True)

# Save files
print("\nPreparing files for archive...")

# 1. Save split metadata
metadata_path = temp_dir / "split_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(split_metadata, f, indent=2)
print(f"✓ Saved: {metadata_path}")

# 2. Save train indices
train_indices_path = temp_dir / "train_indices.npy"
np.save(train_indices_path, np.array(train_indices))
print(f"✓ Saved: {train_indices_path}")

# 3. Save validation indices
val_indices_path = temp_dir / "val_indices.npy"
np.save(val_indices_path, np.array(val_indices))
print(f"✓ Saved: {val_indices_path}")

# 4. Save train samples CSV
train_csv_path = temp_dir / "train_samples.csv"
train_samples_df.to_csv(train_csv_path, index=False)
print(f"✓ Saved: {train_csv_path}")

# 5. Save validation samples CSV
val_csv_path = temp_dir / "val_samples.csv"
val_samples_df.to_csv(val_csv_path, index=False)
print(f"✓ Saved: {val_csv_path}")

# 6. Create README
readme_path = temp_dir / "README.txt"
readme_content = f"""VinDr-Mammo Stratified Dataset Split
=====================================

Created: {split_metadata['created_at']}
Split Type: Breast-level stratified (patient-wise)
Random Seed: {RANDOM_STATE}
Train Ratio: {TRAIN_RATIO}

Files Included:
---------------
1. split_metadata.json - Complete split configuration and statistics
2. train_indices.npy - NumPy array of training sample indices
3. val_indices.npy - NumPy array of validation sample indices
4. train_samples.csv - Training samples with metadata
5. val_samples.csv - Validation samples with metadata

Usage:
------
# Load indices
train_indices = np.load('train_indices.npy')
val_indices = np.load('val_indices.npy')

# Or load samples directly
train_df = pd.read_csv('train_samples.csv')
val_df = pd.read_csv('val_samples.csv')

Statistics:
-----------
Total samples: {split_metadata['total_samples']}
Train samples: {split_metadata['train_samples']} ({split_metadata['train_label_counts']['benign']} benign, {split_metadata['train_label_counts']['malignant']} malignant)
Val samples: {split_metadata['val_samples']} ({split_metadata['val_label_counts']['benign']} benign, {split_metadata['val_label_counts']['malignant']} malignant)

Note:
-----
This is a FIXED split. Use the same split for all experiments to ensure reproducibility.
The split is done at the breast level, keeping CC and MLO views from the same breast together.
"""

with open(readme_path, 'w') as f:
    f.write(readme_content)
print(f"✓ Saved: {readme_path}")

In [ ]:
# Create ZIP archive
print(f"\nCreating ZIP archive: {OUTPUT_ZIP}")

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in temp_dir.iterdir():
        if file_path.is_file():
            arcname = file_path.name
            zipf.write(file_path, arcname=arcname)
            print(f"  Added: {arcname}")

print(f"\n✓ Archive created: {OUTPUT_ZIP}")
print(f"  Size: {os.path.getsize(OUTPUT_ZIP) / 1024:.2f} KB")

In [ ]:
# Clean up temporary directory
import shutil
shutil.rmtree(temp_dir)
print("\n✓ Cleaned up temporary files")

## Step 4: Verify Archive Contents

In [ ]:
# List contents of the created archive
print(f"\nVerifying archive contents: {OUTPUT_ZIP}")
print("="*50)

with zipfile.ZipFile(OUTPUT_ZIP, 'r') as zipf:
    for info in zipf.infolist():
        print(f"{info.filename:30s} {info.file_size:>10,} bytes")

print("\n✓ Archive created successfully!")
print(f"\nYou can now share or backup: {OUTPUT_ZIP}")

## Optional: Test Loading from Archive

In [ ]:
# Test loading the split from the archive
print("Testing archive extraction and loading...")

with zipfile.ZipFile(OUTPUT_ZIP, 'r') as zipf:
    # Load metadata
    with zipf.open('split_metadata.json') as f:
        loaded_metadata = json.load(f)
    
    # Extract indices to temporary location
    zipf.extract('train_indices.npy', path='temp_test')
    zipf.extract('val_indices.npy', path='temp_test')
    
    loaded_train_indices = np.load('temp_test/train_indices.npy')
    loaded_val_indices = np.load('temp_test/val_indices.npy')

print(f"\n✓ Loaded metadata: {loaded_metadata['total_samples']} total samples")
print(f"✓ Loaded train indices: {len(loaded_train_indices)}")
print(f"✓ Loaded val indices: {len(loaded_val_indices)}")

# Verify they match
assert np.array_equal(loaded_train_indices, train_indices), "Train indices mismatch!"
assert np.array_equal(loaded_val_indices, val_indices), "Val indices mismatch!"

print("\n✓ All checks passed!")

# Clean up test extraction
shutil.rmtree('temp_test')